# Phase 2G + Phase 4 — DeBERTa cross-encoder, hybrid, ablations, LLM baseline (Colab / GPU)

Runs the **deferred torch slice** of the Explainable-Hybrid-ASAG project end-to-end and produces every
artifact the manuscript's `\TBD{}` / `TODO(after Colab run)` slots depend on:

1. **Phase 2G** — standalone DeBERTa-v3 cross-encoder vs the GBM head (`reports/phase2g/`).
2. **Phase 4 hybrid** — out-of-fold neural signals `neural_oof.parquet` per dataset (row-aligned, no leakage).
3. **Downstream** (CPU) — `ablations` (branch **-D/only-D**), `neural_compare` (three-way, Table 3),
   `train2d` (hybrid tuned + significance), `xai` (SHAP now attributes to `neural_*`).
4. **LLM zero-shot baseline** — the 2026 reviewer expectation (`reports/phase_llm/`).

### Before you run
- **Runtime → Change runtime type → GPU** (T4 works; L4/A100 much faster).
- Run cells **top to bottom**. Stages 2G/OOF are **resumable** — a disconnect only re-does unfinished datasets.
- Rough GPU time, 6 datasets, LoRA, 3 seeds: **~2–3 h on A100/L4, ~5–8 h on a free T4.**
  To go faster first, set `DATASETS = ['mohler']` in the driver cell and scale up.

> Your uploaded zip already contains the tiny `data/processed/*/{encoder,features}.parquet` (4.2 MB total),
> so **no dataset re-download and no SBERT re-encode** are needed here.

In [ ]:
# 1) Dependencies the neural + hybrid + LLM slices need, WITHOUT disturbing Colab's torch/transformers/numpy.
#    Colab already ships torch (CUDA) + transformers; we only add what is missing. lightgbm/optuna are
#    pinned to the project's versions so the GBM re-runs reproduce the paper's numbers.
%pip -q install -U gdown
%pip -q install loguru peft sentencepiece protobuf accelerate lightgbm==4.5.0 optuna==4.1.0
print('installs complete — if pip printed a dependency-resolver warning, it is safe to ignore here')

In [ ]:
# 2) Download your project zip from Google Drive and locate the repo root robustly.
import os, sys, glob, subprocess, shutil
FILE_ID = '1KjCkzbe_Y49B8ilGFCbCLZ4qQORNxUbY'   # <-- your Drive file id
ZIP = '/content/asag_project.zip'
if not os.path.exists(ZIP):
    subprocess.run(['gdown', FILE_ID, '-O', ZIP], check=True)
shutil.rmtree('/content/extracted', ignore_errors=True)
subprocess.run(['unzip', '-q', '-o', ZIP, '-d', '/content/extracted'], check=True)
cands = [os.path.dirname(p) for p in glob.glob('/content/extracted/**/pyproject.toml', recursive=True)]
REPO = next((d for d in cands if os.path.exists(os.path.join(d, 'configs', 'data.yaml'))), None)
assert REPO, f'could not find repo root under /content/extracted; candidates={cands}'
print('REPO =', REPO)

In [ ]:
# 3) Put src/ on the path + point config at the repo (we do NOT pip-install the project:
#    its requires-python is <3.12 and the Arabic source path breaks editable installs).
os.environ['ASAG_PROJECT_ROOT'] = REPO
os.environ['PYTHONPATH'] = os.path.join(REPO, 'src')
os.environ['PYTHONUTF8'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.chdir(REPO); sys.path.insert(0, os.path.join(REPO, 'src'))
proc = os.path.join(REPO, 'data', 'processed')
need = ('encoder.parquet', 'features.parquet')
present = sorted(d for d in os.listdir(proc)
                 if os.path.isdir(os.path.join(proc, d))
                 and all(os.path.exists(os.path.join(proc, d, f)) for f in need))
assert present, 'no processed parquets in the zip — expected data/processed/<ds>/{encoder,features}.parquet'
print('datasets with encoder+features parquets:', present)

In [ ]:
# 4) Patch configs/data.yaml with a GPU neural block so EVERY entrypoint (CLI / `python -m`) is correct
#    and reproducible — not just an ad-hoc notebook mutation. Idempotent: re-running just resets the block.
import yaml, torch
cfg_path = os.path.join(REPO, 'configs', 'data.yaml')
doc = yaml.safe_load(open(cfg_path, encoding='utf-8'))
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
bs = 32 if any(k in gpu for k in ('A100', 'L4', 'H100', 'A10', 'L40')) else 16   # T4 -> 16
doc['neural'] = {
    'enabled': True, 'backbone': 'microsoft/deberta-v3-base', 'device': 'cuda',
    'epochs': 4, 'batch_size': bs, 'lr': 2.0e-5, 'warmup_ratio': 0.1, 'dropout': 0.1,
    'ordinal_head': 'corn', 'pooling': 'cls',
    'seeds': [42, 1, 2],            # 3 seeds -> publication-grade mean +/- std
    'num_workers': 2, 'select_on_dev': True, 'save_predictions': True,
    'lora_enabled': True, 'lora_r': 8, 'lora_alpha': 16, 'lora_dropout': 0.1,   # protects small corpora
    'max_len': 128, 'max_len_overrides': {'saf': 256},
}
yaml.safe_dump(doc, open(cfg_path, 'w', encoding='utf-8'), sort_keys=False, allow_unicode=True)
print(f'GPU: {gpu} | batch_size={bs} | seeds={doc["neural"]["seeds"]} | LoRA on | epochs=4')

In [ ]:
# 5) Fail-fast: verify the exact APIs the run needs resolve on THIS Colab BEFORE the long job.
import transformers, peft, lightgbm, optuna
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup  # stable 4.x->5.x
print('python', sys.version.split()[0], '| torch', torch.__version__, '| cuda', torch.cuda.is_available())
print('transformers', transformers.__version__, '| peft', peft.__version__,
      '| lightgbm', lightgbm.__version__, '| optuna', optuna.__version__)
assert torch.cuda.is_available(), 'No GPU! Runtime -> Change runtime type -> GPU, then re-run from cell 1.'
# deberta-v3 ships no tokenizer.json -> the fast tokenizer is built from SentencePiece via protobuf.
tok = AutoTokenizer.from_pretrained('microsoft/deberta-v3-base')
enc = tok('a ref [SEP] b', 'student', truncation=True, max_length=16, padding='max_length', return_tensors='pt')
assert enc['input_ids'].shape[1] == 16
import asag
from asag.config import load_data_config
load_data_config.cache_clear()
cfg = load_data_config()
assert cfg.neural.device == 'cuda' and cfg.neural.lora_enabled
print('asag OK from', asag.__file__)
print('neural.device =', cfg.neural.device, '| lora =', cfg.neural.lora_enabled,
      '| seeds =', cfg.neural.seeds, '| deberta-v3 tokenizer OK')
print('\u2713 no conflicts — APIs resolve on this runtime')

## Driver
`DATASETS` is processed small → large so early progress is visible and a free-T4 disconnect costs the least.
Edit it to run a subset / resume. Every stage below skips work whose output already exists on disk.

In [ ]:
DATASETS = ['mohler', 'powergrading', 'mindreading', 'saf', 'asap_sas', 'semeval']
DATASETS = [d for d in DATASETS if d in present]
def run(*mod_args):
    print('\u00bb python -m', *mod_args, flush=True)
    return subprocess.run([sys.executable, '-m', *mod_args], cwd=REPO).returncode
print('will process:', DATASETS)

In [ ]:
# 6) Phase 2G — standalone DeBERTa vs GBM (resumable via a per-dataset partial cache).
#    This is the heaviest stage: n_test_splits (or 5 folds) x 3 seeds full fine-tunes per dataset.
import json
from asag.neural import NEURAL_SCHEMA_VERSION
from asag.neural.run import _write_figure
p2g = os.path.join(REPO, 'reports', 'phase2g'); part = os.path.join(p2g, '_partial')
os.makedirs(part, exist_ok=True)
for ds in DATASETS:
    cache = os.path.join(part, ds + '.json')
    if os.path.exists(cache):
        print('skip (cached):', ds); continue
    if run('asag.neural.run', ds) == 0 and os.path.exists(os.path.join(p2g, 'results.json')):
        block = json.load(open(os.path.join(p2g, 'results.json'))).get('datasets', {}).get(ds)
        if block:
            json.dump(block, open(cache, 'w'), default=str)
blocks = {ds: json.load(open(os.path.join(part, ds + '.json')))
          for ds in DATASETS if os.path.exists(os.path.join(part, ds + '.json'))}
if blocks:
    json.dump({'schema_version': NEURAL_SCHEMA_VERSION, 'backbone': cfg.neural.backbone,
               'seeds': list(cfg.neural.seeds), 'datasets': blocks},
              open(os.path.join(p2g, 'results.json'), 'w'), indent=2, default=str)
    _write_figure(cfg, blocks)
    print('Phase 2G complete for:', list(blocks))

In [ ]:
# 7) Phase 4 hybrid — out-of-fold DeBERTa signals -> data/processed/<ds>/neural_oof.parquet.
#    This is the critical artifact (feeds ablations -D, three-way, train2d, xai). Resumable per dataset.
for ds in DATASETS:
    oof = os.path.join(REPO, 'data', 'processed', ds, 'neural_oof.parquet')
    if os.path.exists(oof):
        print('skip (exists):', ds); continue
    run('asag.neural.extract_features', ds)

In [ ]:
# 8) Verify OOF coverage — every row must get an out-of-fold prediction (~100%).
import pandas as pd, numpy as np
for ds in DATASETS:
    oof = os.path.join(REPO, 'data', 'processed', ds, 'neural_oof.parquet')
    if os.path.exists(oof):
        d = pd.read_parquet(oof)
        print(f'{ds:14s} rows={len(d):6d}  OOF coverage={np.isfinite(d["neural_score"]).mean():.1%}')
    else:
        print(f'{ds:14s} MISSING neural_oof.parquet — re-run stage 7 for this dataset')

In [ ]:
# 9) Downstream GBM-with-neural (CPU, fast). load_bundle auto-concatenates neural_* -> hybrid everywhere.
run('asag.models.ablations')        # branch D: -D / only-D now populated
run('asag.models.neural_compare')   # three-way neural-only / feature-only / hybrid -> reports/phase_hybrid/
run('asag.models.train2d')          # hybrid tuned headline + paired-bootstrap significance
run('asag.xai.run')                 # SHAP now attributes to neural_* (the attribution-dilution analysis)
print('downstream complete')

## LLM zero-shot baseline
Grades each dataset's headline split under the **same** metrics, temperature 0, prompts logged. Default model
`Qwen/Qwen2.5-3B-Instruct` fits a T4; set `LLM_MODEL` to a 7B/8B open model (needs more VRAM) or wire an API
inside `generate()` for the headline row. The neural subprocesses above have exited, so the GPU is free.

In [ ]:
# 10) Write the LLM baseline script into the repo (your uploaded zip predates it), then run it.
llm_code = '"""Zero-shot LLM grading baseline under the paper\'s own metrics/protocol.\n\nGrades each dataset\'s **headline split** with an instruction-tuned LLM (temperature\n0, prompts logged), then scores it with the *same* metric the GBM/DeBERTa arms use\n(``asag.models.metrics`` via ``asag.models.tasks.REGISTRY``). This is the LLM row\nreviewers expect in 2026; the project\'s own review simulation flags its absence as a\nnear-reject at Q2 (Reviewer C).\n\nDesign choices (all deliberate, all documented so a reviewer can check):\n\n* **Zero-shot, no training, no folds.** ``official_split`` datasets are graded on the\n  last / hardest test split (SemEval ``test_ud``, SAF ``test_uq``, ASAP-SAS\n  ``test_ua``); ``kfold`` datasets (Mohler, Powergrading, MIND-CA) are graded once on\n  all CV rows — there is nothing to fit, so the fold structure is irrelevant.\n* **Task-aware prompt + metric.** classification → pick one label (macro-F1 on shared\n  codes); ordinal → an integer in the observed range (QWK); regression → a number in\n  range (Pearson). Per-prompt datasets (ASAP-SAS) average the metric across prompts,\n  exactly like the GBM headline.\n* **Cost cap.** ``LLM_MAX_ROWS`` (default 400) subsamples large test splits with a\n  fixed seed; ``n_eval`` is reported so the estimate\'s basis is explicit.\n\nSwap ``LLM_MODEL`` for a stronger open model (7B/8B) or wire an API inside\n``generate()`` for the headline row; the metric plumbing is unchanged.\n\n    python experiments/llm_zeroshot_baseline.py [dataset ...]\n    LLM_MODEL=Qwen/Qwen2.5-3B-Instruct LLM_MAX_ROWS=400 python experiments/llm_zeroshot_baseline.py\n\nOutputs: reports/phase_llm/{llm_baseline.json, examples.json}.\n"""\nfrom __future__ import annotations\n\nimport json\nimport os\nimport re\nimport sys\n\nimport numpy as np\nimport pandas as pd\n\nfrom asag.config import load_data_config\nfrom asag.models.metrics import compute_metrics\nfrom asag.models.tasks import REGISTRY, get_spec\n\nMODEL = os.environ.get("LLM_MODEL", "Qwen/Qwen2.5-3B-Instruct")\nMAX_ROWS = int(os.environ.get("LLM_MAX_ROWS", "400"))          # cap eval rows/dataset (0 = all)\nMAX_PER_PROMPT = int(os.environ.get("LLM_MAX_PER_PROMPT", "150"))\nBATCH = int(os.environ.get("LLM_BATCH", "16"))\nSEED = 42\n\n\ndef _eval_rows(df: pd.DataFrame, spec) -> pd.DataFrame:\n    """Headline-split rows for official_split; all CV rows for kfold. Cost-capped."""\n    if spec.protocol == "kfold":\n        mask = pd.to_numeric(df["fold"], errors="coerce").fillna(-1) >= 0\n    else:\n        mask = df["split"].astype(str) == spec.test_splits[-1]\n    d = df[mask].reset_index(drop=True)\n    if spec.per_prompt and MAX_PER_PROMPT:\n        d = (d.groupby("question_id", group_keys=False)\n               .apply(lambda g: g.sample(n=min(len(g), MAX_PER_PROMPT), random_state=SEED))\n               .reset_index(drop=True))\n    elif MAX_ROWS and len(d) > MAX_ROWS:\n        d = d.sample(n=MAX_ROWS, random_state=SEED).reset_index(drop=True)\n    return d\n\n\ndef _label_space(df: pd.DataFrame, spec) -> dict:\n    """Allowed labels (classification) or numeric range (ordinal/regression)."""\n    if spec.task_type == "classification":\n        labs = sorted(s for s in df["label"].astype(str).unique() if s and s.lower() != "nan")\n        return {"labels": labs, "vocab": {lab: i for i, lab in enumerate(labs)}}\n    s = pd.to_numeric(df["score"], errors="coerce").dropna()\n    return {"lo": float(s.min()), "hi": float(s.max())}\n\n\ndef _prompt(row: pd.Series, spec, space: dict) -> str:\n    q = str(row.get("question_enc") or "").strip()\n    ref = str(row.get("reference_answer_enc") or "").strip()\n    ans = str(row.get("student_answer_enc") or "").strip()\n    ctx = ""\n    if q and q.lower() != "nan":\n        ctx += f"Question: {q}\\n"\n    if ref and ref.lower() != "nan":\n        ctx += f"Reference answer: {ref}\\n"\n    if spec.task_type == "classification":\n        task = (f"Classify the student answer into exactly one of these labels: "\n                f"{\', \'.join(space[\'labels\'])}.\\nReply with ONLY the label, nothing else.")\n    elif spec.task_type == "ordinal":\n        task = (f"Grade the student answer with an INTEGER score from {int(space[\'lo\'])} to "\n                f"{int(space[\'hi\'])} (higher = better). Reply with ONLY the integer.")\n    else:\n        task = (f"Grade the student answer with a score from {space[\'lo\']:.2f} to "\n                f"{space[\'hi\']:.2f} (higher = better). Reply with ONLY the number.")\n    return f"{ctx}Student answer: {ans}\\n\\n{task}"\n\n\ndef _parse(text: str, spec, space: dict) -> float:\n    t = (text or "").strip()\n    if spec.task_type == "classification":\n        low = t.lower()\n        for lab in space["labels"]:                       # first label mentioned wins\n            if lab.lower() in low:\n                return float(space["vocab"][lab])\n        return float(space["vocab"][space["labels"][0]])  # fallback: first label\n    m = re.search(r"-?\\d+(?:\\.\\d+)?", t)\n    mid = (space["lo"] + space["hi"]) / 2.0\n    if not m:\n        return round(mid) if spec.task_type == "ordinal" else mid\n    v = max(space["lo"], min(space["hi"], float(m.group())))\n    return float(round(v)) if spec.task_type == "ordinal" else v\n\n\ndef load_llm():\n    import torch\n    from transformers import AutoModelForCausalLM, AutoTokenizer\n\n    tok = AutoTokenizer.from_pretrained(MODEL)\n    if tok.pad_token is None:\n        tok.pad_token = tok.eos_token\n    tok.padding_side = "left"                              # correct batched decoder-only gen\n    model = AutoModelForCausalLM.from_pretrained(\n        MODEL, torch_dtype=torch.float16,\n        device_map="cuda" if torch.cuda.is_available() else "cpu")\n    model.eval()\n    return tok, model\n\n\ndef generate(tok, model, prompts: list[str]) -> list[str]:\n    import torch\n\n    sys_msg = "You are a strict grading assistant. Follow the requested output format exactly."\n    texts = [tok.apply_chat_template(\n        [{"role": "system", "content": sys_msg}, {"role": "user", "content": p}],\n        tokenize=False, add_generation_prompt=True) for p in prompts]\n    out: list[str] = []\n    for i in range(0, len(texts), BATCH):\n        chunk = texts[i:i + BATCH]\n        enc = tok(chunk, return_tensors="pt", padding=True, truncation=True,\n                  max_length=1024).to(model.device)\n        with torch.no_grad():\n            gen = model.generate(**enc, max_new_tokens=12, do_sample=False,   # temp 0 (greedy)\n                                 pad_token_id=tok.pad_token_id)\n        for j in range(len(chunk)):\n            out.append(tok.decode(gen[j][enc["input_ids"].shape[1]:], skip_special_tokens=True))\n        print(f"    generated {min(i + BATCH, len(texts))}/{len(texts)}", flush=True)\n    return out\n\n\ndef score_dataset(name: str, cfg, tok, model, examples: dict) -> dict:\n    spec = get_spec(name)\n    path = cfg.paths.processed / name / "encoder.parquet"\n    if not path.exists():\n        return {"status": "no_encoder_parquet"}\n    df = pd.read_parquet(path).reset_index(drop=True)\n    space = _label_space(df, spec)\n    d = _eval_rows(df, spec)\n    if d.empty:\n        return {"status": "no_eval_rows"}\n\n    prompts = [_prompt(r, spec, space) for _, r in d.iterrows()]\n    raw = generate(tok, model, prompts)\n    y_pred = np.array([_parse(t, spec, space) for t in raw], dtype=float)\n    if spec.task_type == "classification":\n        y_true = d["label"].astype(str).map(space["vocab"]).to_numpy(dtype=float)\n    else:\n        y_true = pd.to_numeric(d["score"], errors="coerce").to_numpy(dtype=float)\n    fin = np.isfinite(y_true) & np.isfinite(y_pred)\n    examples[name] = [{"prompt": prompts[k], "llm_raw": raw[k], "pred": float(y_pred[k]),\n                       "true": float(y_true[k])} for k in range(min(3, len(prompts)))]\n\n    def metric(mask: np.ndarray) -> float:\n        return compute_metrics(y_true[mask], y_pred[mask], (spec.headline,)).get(\n            spec.headline, float("nan"))\n\n    if spec.per_prompt:\n        qid = d["question_id"].astype(str).to_numpy()\n        vals = [v for p in np.unique(qid[fin]) if np.isfinite(v := metric(fin & (qid == p)))]\n        headline = float(np.mean(vals)) if vals else float("nan")\n    else:\n        headline = metric(fin)\n    return {"status": "ok", "metric": spec.headline,\n            "headline_split": spec.test_splits[-1] if spec.protocol == "official_split" else "cv",\n            "n_eval": int(fin.sum()),\n            "llm_headline": None if not np.isfinite(headline) else round(float(headline), 4),\n            "model": MODEL}\n\n\ndef main(names: list[str]) -> None:\n    cfg = load_data_config()\n    names = names or [n for n in REGISTRY if (cfg.paths.processed / n / "encoder.parquet").exists()]\n    tok, model = load_llm()\n    out, examples = {}, {}\n    for n in names:\n        print(f"=== LLM zero-shot: {n} ({MODEL}) ===", flush=True)\n        out[n] = score_dataset(n, cfg, tok, model, examples)\n        print(f"    {n}: {out[n]}", flush=True)\n    d = cfg.paths.reports / "phase_llm"\n    d.mkdir(parents=True, exist_ok=True)\n    (d / "llm_baseline.json").write_text(json.dumps(\n        {"model": MODEL, "max_rows": MAX_ROWS, "temperature": 0.0, "datasets": out},\n        indent=2, default=str), encoding="utf-8")\n    (d / "examples.json").write_text(json.dumps(examples, indent=2, default=str), encoding="utf-8")\n    print(f"wrote {d / \'llm_baseline.json\'}")\n\n\nif __name__ == "__main__":\n    main([a for a in sys.argv[1:] if not a.startswith("-")])\n'
dst = os.path.join(REPO, 'experiments', 'llm_zeroshot_baseline.py')
os.makedirs(os.path.dirname(dst), exist_ok=True)
open(dst, 'w', encoding='utf-8').write(llm_code)
print('wrote', dst, '(', len(llm_code), 'bytes )')

In [ ]:
# 11) Run the zero-shot baseline (temp 0). Capped at LLM_MAX_ROWS rows/dataset for cost; raise for full sets.
env = dict(os.environ, LLM_MODEL='Qwen/Qwen2.5-3B-Instruct', LLM_MAX_ROWS='400', LLM_BATCH='16')
subprocess.run([sys.executable, 'experiments/llm_zeroshot_baseline.py', *DATASETS], cwd=REPO, env=env)
import json
p = os.path.join(REPO, 'reports', 'phase_llm', 'llm_baseline.json')
if os.path.exists(p):
    print(json.dumps(json.load(open(p)), indent=2)[:2000])

In [ ]:
# 12) Bundle every new / updated artifact and download it. Unzip it over your local repo root to merge.
OUTDIR = '/content/results_bundle'
shutil.rmtree(OUTDIR, ignore_errors=True); os.makedirs(OUTDIR)
patterns = ['data/processed/*/neural_oof.parquet',
            'reports/phase2g/results.*', 'reports/phase2g/preds/*',
            'reports/phase_hybrid/*', 'reports/phase_llm/*',
            'reports/phase3/*', 'reports/phase2d/*', 'reports/phase2f/*',
            'reports/figures/*.png']
n = 0
for pat in patterns:
    for f in glob.glob(os.path.join(REPO, pat), recursive=True):
        if os.path.isfile(f):
            rel = os.path.relpath(f, REPO); dstf = os.path.join(OUTDIR, rel)
            os.makedirs(os.path.dirname(dstf), exist_ok=True); shutil.copy2(f, dstf); n += 1
print(n, 'files bundled')
shutil.make_archive('/content/results_bundle', 'zip', OUTDIR)
from google.colab import files; files.download('/content/results_bundle.zip')

## After download — fill the paper
1. Unzip `results_bundle.zip` over your local repo root (it merges `neural_oof.parquet` + the new reports).
2. Key numbers now available:
   - `reports/phase_hybrid/three_way.json` → Table 3 (`tab:hybrid`), RQ5 section, abstract sentence.
   - `reports/phase2g/results.json` → neural-only column.
   - `reports/phase3/ablations.json` → the `-D` / `only-D` rows.
   - `reports/phase2f/shap.json` → share of global |SHAP| absorbed by `neural_*` (attribution dilution).
   - `reports/phase_llm/llm_baseline.json` → the LLM baseline row + discussion.
3. Run `python paper/verify_numbers.py`, then replace each `\TBD{...}` in `paper/main.tex`.

**Still open for a strong Q2 (from your own roadmap):** verify the ~12 flagged bib entries, add the
runtime/memory table, LODO threshold-sensitivity + perturbation CIs, and a native-English editing pass.